# Notebook 8: k-Nearest Neighbors (k-NN)

**Difficulty:** ⭐⭐⭐ | **Estimated Time:** 2.5 hours | **Theme:** Spam Email Detection

---

## Why This Matters

k-NN is often called the "lazy" algorithm because it does zero computation at training time — it simply memorizes the data. At prediction time, it finds the k most similar examples and takes a majority vote.

Despite its simplicity, k-NN teaches three fundamental ML concepts:

1. **Distance is everything** — choice of metric and feature scale completely changes which neighbors are "nearest"
2. **The bias-variance tradeoff** — k=1 overfits; large k underfits; the sweet spot is in between
3. **The curse of dimensionality** — k-NN fails silently in high dimensions because all points become equally far apart

---

## Real-World Analogy First

### Asking Your k Closest Friends for a Recommendation

You receive a suspicious email and you're not sure if it's spam. What do you do?

You describe the email to your friends: *"It has 85 words, 7 exclamation marks, mentions a prize, and came from an unknown sender."*

- **k=1:** You ask your single most similar friend (the one who got the most similar email). Whatever they think, you think. Very fast, very dependent on one opinion — might be wrong if that friend is quirky.
- **k=5:** You ask your 5 closest friends. 4 say spam, 1 says legit → you vote: spam. More robust.
- **k=100:** You ask 100 acquaintances. If 50% of all emails are spam, you'll always say spam regardless of this particular email's content. You've over-smoothed.

**The catch:** "Closest" depends on how you measure similarity. If one friend is similar to you by age (differs by 2 years) and another by income (differs by \$50,000), income dominates if you don't normalize.

---

## Topics Covered

1. k-NN algorithm from scratch  
2. Distance metrics: Euclidean, Manhattan, Cosine  
3. Feature scaling — why it's mandatory  
4. k selection via cross-validation  
5. Decision boundary visualization  
6. Curse of dimensionality  
7. Time complexity and practical solutions

## Algorithm: k-NN in Plain English

**Training phase:** `O(1)` — just store the data. Nothing computed.

**Prediction phase for a new email x:**
1. Compute distance from `x` to every training example
2. Sort by distance — find the k smallest
3. Look at the labels of those k neighbors
4. Return the majority vote (for classification) or mean (for regression)

**Time complexity:**
- Training: `O(1)` — just store data
- Prediction: `O(n · d)` per query — compute distance to all `n` training points in `d` dimensions
- For 1 million training emails: each new email requires 1 million distance calculations!

**Solutions for large n:**
- **KD-Tree:** Partition space into a tree; prune branches that can't contain nearest neighbors. Good for d < ~20.
- **Ball Tree:** Uses hyperspheres instead of hyperrectangles; better for curved manifolds and d > 20.
- **Approximate Nearest Neighbors (ANN):** e.g., FAISS, Annoy, HNSW — trade tiny accuracy loss for massive speed gains.

---

## Distance Metrics

| Metric | Formula | Intuition |
|---|---|---|
| **Euclidean (L2)** | $\sqrt{\sum_i (x_i - z_i)^2}$ | Straight-line distance (Pythagoras) |
| **Manhattan (L1)** | $\sum_i |x_i - z_i|$ | City-block distance, robust to outliers |
| **Cosine similarity** | $\frac{x \cdot z}{\|x\| \|z\|}$ | Angle between vectors, ignores magnitude |
| **Minkowski** | $(\sum_i |x_i - z_i|^p)^{1/p}$ | Generalizes Euclidean (p=2) and Manhattan (p=1) |

**For email classification:**
- Euclidean: natural choice for numerical features (word count, length)
- Cosine: natural for text/TF-IDF features (direction matters, not magnitude)

---

## The Curse of Dimensionality

In high dimensions, something strange happens: all points become roughly equidistant from each other.

**Intuition:** In 2D, a random point is sometimes close and sometimes far from another. In 1000D, you're summing 1000 random squared distances — by the law of large numbers, they all converge to nearly the same value. The ratio of max-to-min distance → 1.

**Consequence for k-NN:** Your "nearest" neighbors are no longer meaningfully close — they're just as far as your "farthest" neighbors. The concept of neighborhood breaks down.

In [ ]:
# Cell 1: Imports and setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import time
from collections import Counter
from sklearn.datasets import make_moons, make_classification
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# Reproducibility
np.random.seed(42)

# Plot style
plt.style.use('seaborn-v0_8-whitegrid')
COLORS = {'spam': '#e74c3c', 'legit': '#2ecc71', 'boundary': '#3498db', 'neutral': '#95a5a6'}

print("Libraries loaded successfully.")
print("Notebook 8: k-Nearest Neighbors")
print("Theme: Spam Email Detection")

In [ ]:
# Cell 2: Generate spam email dataset for k-NN experiments
# Features: word_count, exclamation_count, link_count, caps_ratio
# More features than Notebook 7 — lets us explore distance metric effects

np.random.seed(42)
n_samples = 800  # 800 total emails for decent training data

# Legitimate email features (4 features)
n_legit = 500
X_legit = np.column_stack([
    np.random.normal(40, 12, n_legit),      # word_count: moderate, low variance
    np.random.normal(1.2, 0.8, n_legit),    # exclamation_count: rare
    np.random.normal(1.5, 1.0, n_legit),    # link_count: few links
    np.random.beta(2, 8, n_legit) * 0.3,    # caps_ratio: mostly lowercase
])

# Spam email features
n_spam = 300
X_spam = np.column_stack([
    np.random.normal(80, 25, n_spam),        # word_count: longer promotional text
    np.random.normal(6.0, 2.5, n_spam),      # exclamation_count: lots!
    np.random.normal(5.5, 2.0, n_spam),      # link_count: many links
    np.random.beta(5, 3, n_spam) * 0.6,      # caps_ratio: MORE CAPS!!!
])

# Combine and clip to realistic ranges
X = np.vstack([X_legit, X_spam])
y = np.array([0]*n_legit + [1]*n_spam)
X[:, 0] = np.clip(X[:, 0], 1, 250)    # word count
X[:, 1] = np.clip(X[:, 1], 0, 20)     # exclamation count
X[:, 2] = np.clip(X[:, 2], 0, 15)     # link count
X[:, 3] = np.clip(X[:, 3], 0, 1)      # caps ratio

feature_names = ['word_count', 'exclamation_count', 'link_count', 'caps_ratio']
df = pd.DataFrame(X, columns=feature_names)
df['label'] = y

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Dataset: {X.shape[0]} emails, {X.shape[1]} features")
print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")
print(f"\nFeature ranges:")
for i, name in enumerate(feature_names):
    print(f"  {name:25s}: [{X[:, i].min():.2f}, {X[:, i].max():.2f}]")

In [ ]:
# Cell 3: k-NN from scratch — complete implementation
# This is the core algorithm — no sklearn, pure numpy

class KNNClassifier:
    """
    k-Nearest Neighbors Classifier implemented from scratch.
    
    Training: O(1) — just stores the data
    Prediction: O(n * d) per query — brute force distance computation
    """

    def fit(self, X_train, y_train):
        """Store training data — that's it. No parameters learned."""
        self.X_train = np.array(X_train, dtype=float)
        self.y_train = np.array(y_train)
        self.n_classes = len(np.unique(y_train))
        print(f"k-NN fitted: stored {len(X_train)} training examples.")
        print("No model parameters. No optimization. Just memorization.")
        return self

    def _euclidean_distance(self, x1, x2):
        """Compute Euclidean (L2) distance between two vectors."""
        return np.sqrt(np.sum((x1 - x2) ** 2))

    def predict_single(self, x_query, k=5):
        """Predict class for a single query point."""
        # Step 1: Compute distance from query to every training point
        distances = np.array([
            self._euclidean_distance(x_query, x_train)
            for x_train in self.X_train
        ])
        # Step 2: Sort and pick k smallest (argsort gives indices)
        k_nearest_indices = np.argsort(distances)[:k]
        # Step 3: Look up their labels
        k_nearest_labels  = self.y_train[k_nearest_indices]
        # Step 4: Majority vote — most common label wins
        vote_counts = Counter(k_nearest_labels)
        return vote_counts.most_common(1)[0][0]

    def predict(self, X_query, k=5):
        """Predict classes for a batch of query points."""
        return np.array([self.predict_single(x, k) for x in X_query])

# Instantiate and fit
knn_scratch = KNNClassifier()
knn_scratch.fit(X_train, y_train)

In [ ]:
# Cell 4: Verify scratch implementation matches sklearn
# Both should give identical predictions

# Use StandardScaler — k-NN REQUIRES normalized features
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

K = 7  # test with k=7

# Sklearn k-NN for reference
knn_sk = KNeighborsClassifier(n_neighbors=K, metric='euclidean')
knn_sk.fit(X_train_sc, y_train)
sk_preds = knn_sk.predict(X_test_sc)

# Our scratch k-NN (on scaled data)
knn_scratch_sc = KNNClassifier()
knn_scratch_sc.fit(X_train_sc, y_train)

# Predict on a subset (scratch is slow O(n*d) — use first 30 test points)
n_verify = 30
scratch_preds_subset = knn_scratch_sc.predict(X_test_sc[:n_verify], k=K)
sk_preds_subset      = sk_preds[:n_verify]

matches = np.sum(scratch_preds_subset == sk_preds_subset)

print(f"Verifying k-NN implementations (k={K}, first {n_verify} test points):")
print(f"  Scratch predictions: {scratch_preds_subset}")
print(f"  Sklearn predictions: {sk_preds_subset}")
print(f"  Matching predictions: {matches}/{n_verify} = {100*matches/n_verify:.1f}%")
print()
if matches == n_verify:
    print("PERFECT MATCH — our implementation is correct!")
else:
    print(f"Discrepancies: {n_verify - matches} (may be tie-breaking differences)")

# Full accuracy on sklearn
full_acc = accuracy_score(y_test, sk_preds)
print(f"\nFull test accuracy (sklearn, k={K}): {full_acc:.4f}")

In [ ]:
# Cell 5: Distance metrics — compare Euclidean, Manhattan, Cosine on 3 email pairs
# Shows how the same pairs can be "close" or "far" depending on metric

def euclidean_distance(a, b):
    """L2 distance — straight-line distance in feature space."""
    return np.sqrt(np.sum((a - b) ** 2))

def manhattan_distance(a, b):
    """L1 distance — city-block, sum of absolute differences."""
    return np.sum(np.abs(a - b))

def cosine_similarity(a, b):
    """Cosine similarity — 1 means identical direction, 0 means orthogonal."""
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return np.dot(a, b) / (norm_a * norm_b)

# Three pairs of emails (using raw feature values for illustration)
email_A = np.array([42.0, 1.5, 2.0, 0.05])   # typical legit
email_B = np.array([45.0, 1.8, 2.2, 0.06])   # very similar legit
email_C = np.array([85.0, 7.0, 6.0, 0.45])   # typical spam
email_D = np.array([170., 14.0, 12.0, 0.90]) # extreme spam (2x the magnitudes)

pairs = [
    ('Legit A vs Similar Legit B', email_A, email_B),
    ('Legit A vs Spam C',          email_A, email_C),
    ('Spam C vs Extreme Spam D',   email_C, email_D),
]

print(f"{'Email Pair':<35} {'Euclidean':>12} {'Manhattan':>12} {'Cosine Sim':>12}")
print("-" * 75)
for name, ea, eb in pairs:
    ed  = euclidean_distance(ea, eb)
    md  = manhattan_distance(ea, eb)
    cos = cosine_similarity(ea, eb)
    print(f"{name:<35} {ed:>12.3f} {md:>12.3f} {cos:>12.4f}")

print()
print("Key observation: Spam C vs Extreme Spam D have HIGH Euclidean/Manhattan distance")
print("(very different magnitudes) but HIGH Cosine similarity (same direction/pattern).")
print("Cosine similarity is better for text vectors where length varies.")
print()
print("CRITICAL: word_count dominates Euclidean because its scale (1-250)")
print("dwarfs caps_ratio (0-1). THIS is why we must normalize first!")

In [ ]:
# Cell 6: Feature scaling effect — k-NN without vs with StandardScaler
# word_count is in range [1, 250]; caps_ratio is in range [0, 1]
# Without scaling, word_count dominates the distance calculation completely

# Experiment over multiple k values to see the full picture
k_values  = [1, 3, 5, 7, 11, 15, 21]
acc_raw, acc_scaled = [], []

for k in k_values:
    # Unscaled k-NN — large-scale features dominate
    knn_raw = KNeighborsClassifier(n_neighbors=k, metric='euclidean')
    knn_raw.fit(X_train, y_train)
    acc_raw.append(accuracy_score(y_test, knn_raw.predict(X_test)))

    # Scaled k-NN — features are on equal footing
    knn_sc = KNeighborsClassifier(n_neighbors=k, metric='euclidean')
    knn_sc.fit(X_train_sc, y_train)
    acc_scaled.append(accuracy_score(y_test, knn_sc.predict(X_test_sc)))

# Show results as a table
print(f"{'k':>4}  {'Unscaled Acc':>15}  {'Scaled Acc':>12}  {'Improvement':>12}")
print("-" * 50)
for k, raw, sc in zip(k_values, acc_raw, acc_scaled):
    print(f"{k:>4}  {raw:>15.4f}  {sc:>12.4f}  {sc - raw:>+12.4f}")

# Plot the comparison
plt.figure(figsize=(9, 5))
plt.plot(k_values, acc_raw,    'o--', color='#e67e22', label='Without Scaling', linewidth=2.5, markersize=8)
plt.plot(k_values, acc_scaled, 's-',  color=COLORS['boundary'], label='With StandardScaler', linewidth=2.5, markersize=8)
plt.xlabel('k (Number of Neighbors)', fontsize=12)
plt.ylabel('Test Accuracy', fontsize=12)
plt.title('Figure 1: Feature Scaling is MANDATORY for k-NN\n'
          'word_count (range 1-250) dominates caps_ratio (range 0-1) without scaling', fontsize=11)
plt.legend(fontsize=11)
plt.ylim(0.5, 1.0)
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 7: k selection via cross-validation — find the optimal k
# We evaluate k = 1, 3, 5, ..., 29 using 5-fold cross-validation
# k=1: memorizes training, high variance (overfits)
# Large k: smooth boundary, may underfit

k_range    = list(range(1, 30, 2))  # odd k avoids ties: 1, 3, 5, ..., 29
cv_means   = []
cv_stds    = []

for k in k_range:
    knn_cv = KNeighborsClassifier(n_neighbors=k, metric='euclidean')
    # 5-fold cross-validation on the TRAINING set only (never touch test set for tuning!)
    scores = cross_val_score(knn_cv, X_train_sc, y_train, cv=5, scoring='accuracy')
    cv_means.append(scores.mean())
    cv_stds.append(scores.std())

cv_means = np.array(cv_means)
cv_stds  = np.array(cv_stds)

# Find the best k
best_idx = np.argmax(cv_means)
best_k   = k_range[best_idx]
best_acc = cv_means[best_idx]

# Plot cross-validation curve
plt.figure(figsize=(10, 5))
plt.plot(k_range, cv_means, 'o-', color=COLORS['boundary'], linewidth=2.5, markersize=7, label='CV Mean Accuracy')
plt.fill_between(k_range, cv_means - cv_stds, cv_means + cv_stds,
                 alpha=0.2, color=COLORS['boundary'], label='±1 std')
plt.axvline(best_k, color=COLORS['spam'], linestyle='--', linewidth=2.5,
            label=f'Best k={best_k} (acc={best_acc:.4f})')
# Annotate bias-variance zones
plt.annotate('High Variance\n(overfits, memorizes)', xy=(2, cv_means[0]),
             xytext=(4, cv_means[0] - 0.03), fontsize=9, color='#e74c3c',
             arrowprops=dict(arrowstyle='->', color='#e74c3c'))
plt.annotate('High Bias\n(too smooth, underfits)', xy=(27, cv_means[-1]),
             xytext=(18, cv_means[-1] - 0.025), fontsize=9, color='#e67e22',
             arrowprops=dict(arrowstyle='->', color='#e67e22'))
plt.xlabel('k (Number of Neighbors)', fontsize=12)
plt.ylabel('5-Fold CV Accuracy', fontsize=12)
plt.title('Figure 2: k Selection via Cross-Validation\n'
          'k=1 overfits; large k underfits; optimal k is in the sweet spot', fontsize=11)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

print(f"Best k = {best_k} with CV accuracy = {best_acc:.4f}")

In [ ]:
# Cell 8: Decision boundary visualization — 4-panel subplot for k=1, 3, 10, 50
# Using make_moons for a non-linear dataset where boundary evolution is dramatic
# This clearly shows the bias-variance tradeoff as k increases

# Generate 2D moons dataset (non-linearly separable)
X_moons, y_moons = make_moons(n_samples=300, noise=0.25, random_state=42)
X_moons_sc = StandardScaler().fit_transform(X_moons)

# Create a dense grid for decision boundary plotting
x_min, x_max = X_moons_sc[:, 0].min() - 0.5, X_moons_sc[:, 0].max() + 0.5
y_min, y_max = X_moons_sc[:, 1].min() - 0.5, X_moons_sc[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 250),
                     np.linspace(y_min, y_max, 250))

k_vals_plot = [1, 3, 10, 50]
fig, axes   = plt.subplots(2, 2, figsize=(13, 10))
axes        = axes.ravel()

for ax, k in zip(axes, k_vals_plot):
    knn_plot = KNeighborsClassifier(n_neighbors=k)
    knn_plot.fit(X_moons_sc, y_moons)

    # Predict on the full grid
    Z = knn_plot.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    # Background colors for each class region
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdYlGn_r', levels=[-0.5, 0.5, 1.5])
    # Decision boundary line
    ax.contour(xx, yy, Z, levels=[0.5], colors=[COLORS['boundary']], linewidths=2)

    # Plot training points
    for cls, color, name in [(0, COLORS['legit'], 'Legit'), (1, COLORS['spam'], 'Spam')]:
        mask = y_moons == cls
        ax.scatter(X_moons_sc[mask, 0], X_moons_sc[mask, 1],
                   c=color, s=30, alpha=0.7, edgecolors='white', linewidth=0.5, label=name)

    # Compute train and test accuracy for annotation
    train_acc = accuracy_score(y_moons, knn_plot.predict(X_moons_sc))
    description = {1: 'Overfit: jagged boundary\nzero training error',
                   3: 'Still complex boundary\nsome smoothing',
                   10: 'Good boundary\nbias-variance sweet spot',
                   50: 'Underfit: over-smoothed\nhigh bias'}
    ax.set_title(f'k = {k}  |  Train acc = {train_acc:.3f}\n{description[k]}',
                 fontsize=11, fontweight='bold')
    ax.set_xlabel('Feature 1 (scaled)', fontsize=9)
    ax.set_ylabel('Feature 2 (scaled)', fontsize=9)
    ax.legend(fontsize=8, loc='upper right')

plt.suptitle('Figure 3: k-NN Decision Boundaries on make_moons Dataset\n'
             'k=1: memorizes everything | Large k: loses detail',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 9: Curse of Dimensionality — empirical demonstration
# In high dimensions, all pairwise distances converge → k-NN breaks down
# We measure: ratio = max_distance / min_distance for 500 random point pairs
# As d → ∞, ratio → 1 (can't distinguish near from far)

np.random.seed(42)
dimensions  = [2, 5, 10, 50, 100, 200]  # dimensionalities to test
n_points    = 500                         # 500 random points per experiment
n_ref_pairs = 1000                        # 1000 reference pairs for distance stats

ratios_mean   = []   # mean(max_dist / min_dist) across many reference points
relative_stds = []   # std / mean of distances — measures concentration

for d in dimensions:
    # Generate n_points in the d-dimensional unit hypercube
    points = np.random.uniform(0, 1, size=(n_points, d))

    ratio_list = []
    for _ in range(n_ref_pairs):
        # Pick one reference point
        ref_idx = np.random.randint(n_points)
        ref     = points[ref_idx]
        others  = np.delete(points, ref_idx, axis=0)

        # Compute Euclidean distances from ref to all others
        dists = np.sqrt(np.sum((others - ref) ** 2, axis=1))

        # Ratio of max to min distance
        ratio_list.append(dists.max() / (dists.min() + 1e-10))

    ratios_mean.append(np.mean(ratio_list))

    # Also measure coefficient of variation of distances (std/mean)
    all_dists = np.sqrt(np.sum((
        points[np.random.randint(n_points, size=5000)] -
        points[np.random.randint(n_points, size=5000)]
    ) ** 2, axis=1))
    relative_stds.append(all_dists.std() / (all_dists.mean() + 1e-10))

# Display the concentration results
print("Curse of Dimensionality: Distance Concentration")
print("=" * 55)
print(f"{'Dimensions':>12} {'Max/Min Ratio':>15} {'Rel. Std (std/mean)':>20}")
print("-" * 55)
for d, ratio, rstd in zip(dimensions, ratios_mean, relative_stds):
    print(f"{d:>12} {ratio:>15.3f} {rstd:>20.4f}")

print()
print("Max/Min Ratio → 1 means nearest neighbor is as far as farthest neighbor.")
print("Rel. Std → 0 means all distances are nearly equal → 'nearest' is meaningless.")

In [ ]:
# Cell 10: Visualize the curse of dimensionality
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Max/Min ratio vs dimensions
ax1 = axes[0]
ax1.plot(dimensions, ratios_mean, 'o-', color=COLORS['spam'], linewidth=2.5,
         markersize=10, markerfacecolor='white', markeredgewidth=2.5)
ax1.axhline(1.0, color='gray', linestyle='--', linewidth=1.5, label='Ratio=1 (all equidistant)')
for d, r in zip(dimensions, ratios_mean):
    ax1.annotate(f'{r:.1f}', (d, r), textcoords='offset points',
                 xytext=(0, 10), ha='center', fontsize=9, color=COLORS['spam'])
ax1.set_xlabel('Number of Dimensions', fontsize=12)
ax1.set_ylabel('Mean (Max Distance / Min Distance)', fontsize=11)
ax1.set_title('Curse of Dimensionality:\nMax/Min Distance Ratio Collapses to 1',
              fontsize=11, fontweight='bold')
ax1.legend(fontsize=10)
ax1.set_xscale('log')
ax1.grid(True, alpha=0.4)

# Panel 2: Distance histograms at d=2 vs d=100 (showing concentration)
ax2 = axes[1]
np.random.seed(42)
for d, color, label in [(2, COLORS['legit'], 'd=2'), (100, COLORS['spam'], 'd=100')]:
    pts  = np.random.uniform(0, 1, size=(500, d))
    idx1 = np.random.randint(500, size=2000)
    idx2 = np.random.randint(500, size=2000)
    dists = np.sqrt(np.sum((pts[idx1] - pts[idx2])**2, axis=1))
    ax2.hist(dists, bins=40, alpha=0.6, color=color, label=label, density=True)
ax2.set_xlabel('Pairwise Euclidean Distance', fontsize=12)
ax2.set_ylabel('Density', fontsize=12)
ax2.set_title('Distance Histograms: d=2 vs d=100\n'
              'In high-d, all distances concentrate around the same value',
              fontsize=11, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.4)

plt.suptitle('Figure 4: The Curse of Dimensionality\n'
             'k-NN fails in high dimensions — "nearest" becomes meaningless',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 11: Time complexity — measure prediction time vs training set size
# k-NN prediction is O(n * d) — scales linearly with training set size n
# This is the primary practical limitation of brute-force k-NN

import time

# Training sizes to test — from small to very large
train_sizes_timing = [1_000, 5_000, 20_000, 100_000]
n_test_queries     = 100   # predict 100 test emails for timing
d_features         = 10    # simulate 10-dimensional email feature vector
K_timing           = 5

times_brute   = []  # brute-force k-NN timing
times_kdtree  = []  # KD-tree k-NN timing (sklearn default for small d)

for n_train in train_sizes_timing:
    # Generate synthetic training and test data
    X_big_train = np.random.randn(n_train, d_features)
    y_big_train = np.random.randint(0, 2, n_train)
    X_query     = np.random.randn(n_test_queries, d_features)

    # Brute force
    knn_brute = KNeighborsClassifier(n_neighbors=K_timing, algorithm='brute')
    knn_brute.fit(X_big_train, y_big_train)
    t0 = time.perf_counter()
    knn_brute.predict(X_query)
    times_brute.append(time.perf_counter() - t0)

    # KD-tree (more efficient for moderate d)
    knn_kd = KNeighborsClassifier(n_neighbors=K_timing, algorithm='kd_tree')
    knn_kd.fit(X_big_train, y_big_train)
    t0 = time.perf_counter()
    knn_kd.predict(X_query)
    times_kdtree.append(time.perf_counter() - t0)

# Print timing table
print(f"Prediction time for {n_test_queries} queries ({d_features} features, k={K_timing})")
print(f"{'Train Size':>12} {'Brute Force (ms)':>20} {'KD-Tree (ms)':>15} {'Speedup':>10}")
print("-" * 62)
for n, tb, tk in zip(train_sizes_timing, times_brute, times_kdtree):
    speedup = tb / max(tk, 1e-9)
    print(f"{n:>12,} {tb*1000:>20.1f} {tk*1000:>15.1f} {speedup:>9.1f}x")

In [ ]:
# Cell 12: Plot time complexity growth — O(n) for brute force
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Linear scale
ax1 = axes[0]
ax1.plot(train_sizes_timing, [t*1000 for t in times_brute], 'o-',
         color=COLORS['spam'], linewidth=2.5, markersize=9, label='Brute Force O(n·d)')
ax1.plot(train_sizes_timing, [t*1000 for t in times_kdtree], 's--',
         color=COLORS['legit'], linewidth=2.5, markersize=9, label='KD-Tree O(d·log n)')
ax1.set_xlabel('Training Set Size (n)', fontsize=12)
ax1.set_ylabel(f'Prediction Time for {n_test_queries} Queries (ms)', fontsize=11)
ax1.set_title('Prediction Time vs Training Size\n(Linear scale)', fontsize=11)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.4)

# Log-log scale — O(n) shows as a straight line
ax2 = axes[1]
ax2.loglog(train_sizes_timing, [t*1000 for t in times_brute], 'o-',
           color=COLORS['spam'], linewidth=2.5, markersize=9, label='Brute Force O(n·d)')
ax2.loglog(train_sizes_timing, [t*1000 for t in times_kdtree], 's--',
           color=COLORS['legit'], linewidth=2.5, markersize=9, label='KD-Tree O(d·log n)')
ax2.set_xlabel('Training Set Size (n)', fontsize=12)
ax2.set_ylabel(f'Prediction Time (ms)', fontsize=11)
ax2.set_title('Prediction Time vs Training Size\n(Log-Log scale: straight = power law)', fontsize=11)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.4)

plt.suptitle('Figure 5: k-NN Time Complexity — O(n) Bottleneck for Large Datasets\n'
             'Practical solution: KD-Tree (small d) or Approximate NN (large d)',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nFor 100K training emails, brute-force k-NN becomes very slow.")
print("Real solutions:")
print("  1. KD-Tree: O(d * log n) — excellent for d < 20")
print("  2. Ball Tree: better for d > 20 and curved manifolds")
print("  3. FAISS / HNSW: approximate nearest neighbors for d >> 20, n >> 100K")

In [ ]:
# Cell 13: Complete spam classification comparison
# Evaluate k-NN with best k vs other classifiers on our spam dataset

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier

# Use the best k from cross-validation (Cell 7)
classifiers = {
    f'k-NN (k={best_k}, scaled)':   KNeighborsClassifier(n_neighbors=best_k),
    'k-NN (k=1, scaled)':           KNeighborsClassifier(n_neighbors=1),
    'Logistic Regression':           LogisticRegression(max_iter=1000, random_state=42),
    'Naive Bayes':                   GaussianNB(),
    'Decision Tree':                 DecisionTreeClassifier(max_depth=5, random_state=42),
}

print(f"{'Classifier':<35} {'Train Acc':>10} {'Test Acc':>10} {'CV Acc (5-fold)':>16}")
print("-" * 75)

for name, clf in classifiers.items():
    # All classifiers use scaled features
    clf.fit(X_train_sc, y_train)
    train_acc = accuracy_score(y_train, clf.predict(X_train_sc))
    test_acc  = accuracy_score(y_test,  clf.predict(X_test_sc))
    cv_acc    = cross_val_score(clf, X_train_sc, y_train, cv=5).mean()
    print(f"{name:<35} {train_acc:>10.4f} {test_acc:>10.4f} {cv_acc:>16.4f}")

print()
print("Note: k-NN (k=1) has 100% training accuracy (it memorizes everything)")
print("but may not generalize best to test data — this is overfitting!")

In [ ]:
# Cell 14: Final comprehensive summary visualization
# Panel A: k-NN intuition on 2D data
# Panel B: bias-variance summary

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Panel A: k-NN intuition — show the k nearest neighbors for a query point ---
np.random.seed(7)
X_demo = np.vstack([np.random.normal([2, 2], 0.8, (30, 2)),   # legit cluster
                    np.random.normal([5, 5], 0.8, (30, 2))])  # spam cluster
y_demo = np.array([0]*30 + [1]*30)

query = np.array([3.5, 3.5])  # query point in the middle
dists = np.sqrt(np.sum((X_demo - query)**2, axis=1))
k_show = 7
knn_indices = np.argsort(dists)[:k_show]

ax = axes[0]
for cls, color, name in [(0, COLORS['legit'], 'Legitimate'), (1, COLORS['spam'], 'Spam')]:
    mask = y_demo == cls
    ax.scatter(X_demo[mask, 0], X_demo[mask, 1], c=color, s=60, alpha=0.7,
               label=name, edgecolors='white')
# Highlight k nearest neighbors
ax.scatter(X_demo[knn_indices, 0], X_demo[knn_indices, 1],
           s=200, edgecolors='black', linewidths=2.5, facecolors='none',
           zorder=5, label=f'k={k_show} nearest')
# Query point
ax.scatter(*query, c='gold', s=250, zorder=6, edgecolors='black',
           linewidths=2, marker='*', label='Query email')
# Draw lines from query to neighbors
for idx in knn_indices:
    ax.plot([query[0], X_demo[idx, 0]], [query[1], X_demo[idx, 1]],
            'k-', alpha=0.3, linewidth=1)
votes = y_demo[knn_indices]
ax.set_title(f'k-NN with k={k_show}: Query Finds {k_show} Nearest Neighbors\n'
             f'Votes: {np.sum(votes==0)} Legit, {np.sum(votes==1)} Spam → '
             f'Prediction: {"Spam" if votes.sum() > k_show//2 else "Legit"}',
             fontsize=10, fontweight='bold')
ax.legend(fontsize=9)
ax.set_xlabel('Feature 1 (scaled)')
ax.set_ylabel('Feature 2 (scaled)')

# --- Panel B: Bias-Variance Tradeoff Summary ---
ax2 = axes[1]
k_plot_range = np.array(k_range)
ax2.plot(k_plot_range, cv_means, 'o-', color=COLORS['boundary'],
         linewidth=2.5, markersize=6, label='CV Accuracy')
ax2.fill_between(k_plot_range, cv_means - cv_stds, cv_means + cv_stds,
                 alpha=0.15, color=COLORS['boundary'])
ax2.axvline(best_k, color=COLORS['spam'], linestyle='--', linewidth=2,
            label=f'Best k={best_k}')
ax2.set_xlabel('k (Number of Neighbors)', fontsize=12)
ax2.set_ylabel('Cross-Validation Accuracy', fontsize=11)
ax2.set_title('Bias-Variance Tradeoff in k-NN\n'
              'Small k = high variance | Large k = high bias', fontsize=11)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.4)

plt.suptitle('Figure 6: k-NN Summary — Intuition and Hyperparameter Selection',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## k-NN Algorithm Summary

| Aspect | Detail |
|---|---|
| **Type** | Lazy learner (instance-based) — no explicit model |
| **Training time** | O(1) — just stores data |
| **Prediction time** | O(n · d) brute-force; O(d · log n) with KD-tree |
| **Memory** | O(n · d) — must store entire training set |
| **Hyperparameter** | k — number of neighbors |
| **k=1** | Zero training error, high variance, memorizes noise |
| **Large k** | Smooth boundary, high bias, may underfit |
| **Feature scaling** | MANDATORY — large-scale features dominate distance |
| **High dimensions** | Fails due to curse of dimensionality |
| **Best for** | Small datasets, nonlinear boundaries, quick baseline |
| **Avoid when** | n > 100K (too slow), d > 50 (curse of dimensionality) |

## Self-Check Questions

Test your understanding before moving on.

---

**Question 1:** k-NN with k=1 has 0% training error on any dataset. Why? Is this a good thing?

<details>
<summary>Click to reveal answer</summary>

**Answer — Why 0% error:** With k=1, for any training point `x_i`, the nearest neighbor IS `x_i` itself (distance = 0). So the prediction for every training point is its own label — 100% correct by definition.

**Is it good? No.** This is the definition of **overfitting**:
- The model memorizes the training data perfectly, including all label noise
- It has zero training error but can generalize poorly to new emails
- The decision boundary is extremely jagged, following every individual training point
- A single mislabeled email becomes its own island in the decision space

**Analogy:** A student who memorizes every past exam question word-for-word scores 100% on practice but fails on the real exam when questions are slightly rephrased.

**Rule of thumb:** Always use cross-validation to select k. k=1 is almost never the optimal choice unless your data is perfectly clean and very large.

</details>

---

**Question 2:** You have two features: `age` (0–100) and `income` (\$0–\$500,000). Without normalization, which feature dominates the distance calculation, and by how much?

<details>
<summary>Click to reveal answer</summary>

**Answer:** `income` completely dominates.

**Why:** Euclidean distance = $\sqrt{(age_1 - age_2)^2 + (income_1 - income_2)^2}$

- Maximum age difference: 100 → contributes $100^2 = 10,000$ to the squared distance
- Maximum income difference: \$500,000 → contributes $500,000^2 = 250,000,000,000$ to the squared distance
- Income is $250,000,000,000 / 10,000 = 25,000,000\times$ more influential!

**Consequence:** k-NN effectively ignores age entirely. Two emails with identical ages but slightly different incomes will be considered "far apart," while emails with wildly different ages but similar incomes are "neighbors."

**Fix:** Apply `StandardScaler` (zero mean, unit variance) or `MinMaxScaler` (scale to [0,1]) before computing distances. This is mandatory for any distance-based algorithm: k-NN, SVM with RBF kernel, PCA, clustering.

</details>

---

**Question 3:** You have 10 million training emails. k-NN prediction is too slow for real-time spam filtering. What are two practical solutions?

<details>
<summary>Click to reveal answer</summary>

**Answer — Two practical solutions:**

**Solution 1: Tree-based data structures**
- **KD-Tree** (scikit-learn: `algorithm='kd_tree'`): Partitions the feature space into a binary tree. At query time, prunes branches that cannot contain nearest neighbors. Reduces prediction from O(n) to O(log n) for low dimensions (d < ~20).
- **Ball Tree** (`algorithm='ball_tree'`): Uses nested hyperspheres; more efficient than KD-tree for d > 20 or non-Euclidean metrics.
- **Limitation:** Both degrade toward O(n) as d increases (curse of dimensionality)

**Solution 2: Approximate Nearest Neighbors (ANN)**
- Libraries: **FAISS** (Facebook AI), **Annoy** (Spotify), **HNSW** (Hierarchical Navigable Small World graphs)
- Trade a tiny accuracy loss (find 99.9% correct neighbors) for 10-100x speedups
- FAISS can handle billions of vectors with GPU acceleration
- Production recommendation systems (Netflix, Spotify) use ANN, not exact k-NN

**Bonus — Switch algorithms:** For 10M emails, consider switching entirely to Logistic Regression or a small Neural Network. Both have O(d) prediction time regardless of training set size.

</details>

---

**Question 4:** Why does k-NN fail in 1,000 dimensions even with plenty of training data?

<details>
<summary>Click to reveal answer</summary>

**Answer:** This is the **curse of dimensionality**, specifically the **concentration of measure** phenomenon:

**What happens:** In d dimensions, the Euclidean distance between two random points is approximately $\sqrt{d}$ (since each of the d coordinates contributes a random term). As d grows:
- All pairwise distances converge to the same expected value $\sqrt{d}$
- The variance of distances grows much slower than the mean
- The ratio max_distance / min_distance → 1

**Consequence for k-NN:** When all points are roughly equidistant, your "5 nearest neighbors" are just as far away as your "500 nearest neighbors." The concept of a neighborhood becomes meaningless — there is no local structure to exploit.

**Why more data doesn't help:** To maintain the same density of training points as you add dimensions, you'd need exponentially more data. In 2D, 100 points might be dense enough. In 10D, you'd need 100^5 = 10 billion points. In 1000D, no amount of realistic data can cover the space.

**Practical fixes:**
1. **Dimensionality reduction** (PCA, t-SNE, UMAP) before applying k-NN
2. **Feature selection** — only use the most informative features
3. **Switch algorithms** — tree-based models (Random Forests) or linear models handle high dimensions better
4. **Use cosine similarity** instead of Euclidean (partially mitigates concentration for text data)

</details>

In [ ]:
# Cell 15: Final recap and key takeaways
print("=" * 60)
print(" NOTEBOOK 8 SUMMARY: k-Nearest Neighbors")
print("=" * 60)
print()
print("ALGORITHM:")
print("  Train: O(1) — store data")
print("  Predict: find k nearest, majority vote")
print("  Prediction cost: O(n * d) brute force")
print()
print("KEY RULES:")
print("  1. ALWAYS scale features before k-NN")
print("  2. Use cross-validation to select k")
print("  3. k=1 → overfit; large k → underfit")
print("  4. Avoid for d > 50 (curse of dimensionality)")
print("  5. Avoid for n > 100K without ANN indexing")
print()
print("DISTANCE METRICS:")
print("  Euclidean (L2) — default for numerical features")
print("  Manhattan (L1) — more robust to outliers")
print("  Cosine — for text/TF-IDF vectors")
print()
print(f"RESULTS ON SPAM DATASET (4 features, {len(X)} emails):")
knn_final = KNeighborsClassifier(n_neighbors=best_k)
knn_final.fit(X_train_sc, y_train)
final_acc = accuracy_score(y_test, knn_final.predict(X_test_sc))
print(f"  Best k (from CV): {best_k}")
print(f"  Test accuracy   : {final_acc:.4f} ({final_acc*100:.1f}%)")
print()
print("CURSE OF DIMENSIONALITY:")
for d, ratio in zip(dimensions, ratios_mean):
    bar = '#' * int((ratio - 1) * 5)
    print(f"  d={d:>4}: max/min ratio={ratio:.2f} {bar}")
print()
print("Next: Notebook 9 — Naive Bayes Classifier")

In [ ]:
# Cell 16: Weighted k-NN — closer neighbors vote more strongly
# Standard k-NN: every neighbor gets 1 vote regardless of distance
# Weighted k-NN: vote weight = 1 / distance — closer neighbors matter more
# This reduces sensitivity to the exact value of k

# Compare uniform vs distance-weighted voting across k values
k_values_w  = list(range(1, 30, 2))
acc_uniform  = []
acc_weighted = []

for k in k_values_w:
    # Uniform: every neighbor gets equal vote
    knn_u = KNeighborsClassifier(n_neighbors=k, weights='uniform')
    knn_u.fit(X_train_sc, y_train)
    acc_uniform.append(accuracy_score(y_test, knn_u.predict(X_test_sc)))

    # Distance-weighted: closer neighbors count more
    knn_w = KNeighborsClassifier(n_neighbors=k, weights='distance')
    knn_w.fit(X_train_sc, y_train)
    acc_weighted.append(accuracy_score(y_test, knn_w.predict(X_test_sc)))

# Show the comparison table
print(f"{'k':>4}  {'Uniform Vote':>14}  {'Distance-Weighted':>18}  {'Difference':>12}")
print("-" * 55)
for k, u, w in zip(k_values_w, acc_uniform, acc_weighted):
    print(f"{k:>4}  {u:>14.4f}  {w:>18.4f}  {w - u:>+12.4f}")

# Plot both voting schemes
plt.figure(figsize=(10, 5))
plt.plot(k_values_w, acc_uniform,  'o-',  color=COLORS['boundary'], linewidth=2.5,
         markersize=7, label='Uniform voting (1 vote each)')
plt.plot(k_values_w, acc_weighted, 's--', color=COLORS['spam'], linewidth=2.5,
         markersize=7, label='Distance-weighted (1/dist votes)')
plt.xlabel('k (Number of Neighbors)', fontsize=12)
plt.ylabel('Test Accuracy', fontsize=12)
plt.title('Figure 7: Uniform vs Distance-Weighted k-NN Voting\n'
          'Weighted voting often improves accuracy and is more robust to k choice',
          fontsize=11)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

print("\nKey insight: With distance weighting, a very close neighbor dominates.")
print("This reduces the impact of choosing k — curves are smoother.")

In [ ]:
# Cell 17: k-NN for regression — predict email spam score (continuous)
# k-NN isn't limited to classification — it works for regression too
# For regression: instead of majority vote, take the MEAN of k neighbors' values
# Applied to spam: predict a "spam confidence score" (0.0 = clean, 1.0 = definite spam)

from sklearn.neighbors import KNeighborsRegressor

# Create a continuous spam score target (0=definitely legit, 1=definitely spam)
# Simulate this from the true labels + some noise (real-world analog: human rater scores)
np.random.seed(42)
spam_score = y.astype(float) + np.random.normal(0, 0.15, len(y))
spam_score = np.clip(spam_score, 0, 1)  # keep in [0, 1] range

# Split and scale
Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X, spam_score, test_size=0.2, random_state=42
)
scaler_r   = StandardScaler()
Xr_train_sc = scaler_r.fit_transform(Xr_train)
Xr_test_sc  = scaler_r.transform(Xr_test)

# Fit k-NN regressor with best k from earlier
knn_reg = KNeighborsRegressor(n_neighbors=best_k, weights='distance')
knn_reg.fit(Xr_train_sc, yr_train)
yr_pred = knn_reg.predict(Xr_test_sc)

# Evaluate regression quality
from sklearn.metrics import mean_squared_error, r2_score
mse = mean_squared_error(yr_test, yr_pred)
r2  = r2_score(yr_test, yr_pred)

print(f"k-NN Regression (k={best_k}, distance-weighted):")
print(f"  MSE  : {mse:.4f}")
print(f"  RMSE : {np.sqrt(mse):.4f}")
print(f"  R²   : {r2:.4f}")

# Visualize: predicted vs actual spam scores
plt.figure(figsize=(8, 5))
plt.scatter(yr_test, yr_pred, alpha=0.5, color=COLORS['boundary'], s=30, edgecolors='white')
plt.plot([0, 1], [0, 1], 'r--', linewidth=2, label='Perfect prediction')
plt.xlabel('Actual Spam Score', fontsize=12)
plt.ylabel('Predicted Spam Score (k-NN)', fontsize=12)
plt.title(f'Figure 8: k-NN Regression — Predicting Spam Confidence Score\n'
          f'k={best_k}, R²={r2:.3f}, RMSE={np.sqrt(mse):.3f}', fontsize=11)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()
print("\nApplication: k-NN regression can output a SOFT spam score")
print("instead of a binary label — useful for downstream filtering decisions.")

## k-NN Practical Checklist

Before deploying k-NN on any real problem, run through this checklist:

### Before Training
- [ ] **Scale your features** — apply `StandardScaler` or `MinMaxScaler`. Non-negotiable.
- [ ] **Check dimensionality** — if d > 50, consider PCA first to avoid the curse of dimensionality.
- [ ] **Estimate dataset size** — if n > 100K, plan for KD-tree, Ball Tree, or approximate NN.
- [ ] **Choose distance metric** — Euclidean for numerical, Cosine for text/TF-IDF vectors.

### Hyperparameter Selection
- [ ] **Use odd k** to avoid ties in binary classification.
- [ ] **Cross-validate** over k = 1, 3, 5, ..., 29 (or wider) using held-out validation set.
- [ ] **Try distance weighting** (`weights='distance'`) — often improves accuracy with minimal cost.

### Evaluation
- [ ] **Compare train vs test accuracy** — large gap means k is too small (overfitting).
- [ ] **Check prediction time** on your deployment hardware — may need ANN indexing.
- [ ] **Baseline comparison** — k-NN is an excellent quick baseline; compare against LR, Decision Tree.

### Spam Filter Specific
- [ ] **Normalize word_count and link_count** — they have much larger ranges than binary features.
- [ ] **Consider cosine distance** if using TF-IDF bag-of-words features.
- [ ] **Monitor drift** — email patterns change; k-NN must be retrained with fresh data regularly.

---

**Bottom line:** k-NN is the best algorithm for understanding ML fundamentals (distance, neighbors, bias-variance). In production, it's mainly useful as a baseline or for small datasets where interpretability matters — "this email was classified as spam because its 7 most similar training emails were all spam."